# FMP Statement Market Cap Ratios

This notebook tests statement-level market cap ratios built from FMP historical fundamentals. The feature families are `income_mcap`, `balance_mcap`, and `cash_mcap`. The run starts with a `>= $1T` screener universe so the pipeline can be validated on the largest names first.

In [1]:

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

_current = Path.cwd().resolve()
_repo_candidates = [_current, *_current.parents]
REPO_ROOT = next((path for path in _repo_candidates if (path / "quant_warehouse").exists() and (path / "pyproject.toml").exists()), _current)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quant_warehouse import Warehouse
from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.screener_fetch import ScreenerQuery, fetch_equity_screener

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROVIDER = "fmp"
ANALYSIS_LABEL = "OpenBB/FMP screened US >= $1T universe"
SCREEN_MARKET_CAP_MIN = 1_000_000_000_000
SCREEN_COUNTRY = "US"
SCREEN_EXCHANGES = ("NASDAQ", "NYSE")
SCREEN_LIMIT = 5_000
START_DATE = "2018-01-01"
END_DATE = None
FILING_LAG_DAYS = 45
HORIZONS = (20, 60, 120)
MIN_OBS = 40
SOURCE_SECTIONS = ("income", "balance", "cash")
FAMILY_BY_SECTION = {
    "income": "income_mcap",
    "balance": "balance_mcap",
    "cash": "cash_mcap",
}

wh = Warehouse()
run_timings: dict[str, float] = {}


def _read_history_section(symbol: str, section: str) -> pd.DataFrame:
    if section == "prices":
        return wh.read_prices(symbol, provider=PROVIDER)
    return wh.read_fundamentals(symbol, section=section, provider=PROVIDER)


def has_required_warehouse_history(symbol: str) -> tuple[bool, str]:
    required = ("prices", "historical_market_cap", *SOURCE_SECTIONS)
    for section in required:
        try:
            frame = _read_history_section(symbol, section)
        except Exception as exc:
            return False, f"{section}: {type(exc).__name__}"
        if frame is None or frame.empty:
            return False, f"{section}: empty"
        if section in SOURCE_SECTIONS and len(frame) < 8:
            return False, f"{section}: too_few_rows"
    return True, "ok"


def _normalize_screener_frame(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    out = frame.copy()
    rename_map = {
        "companyName": "name",
        "marketCap": "market_cap",
        "exchangeShortName": "exchange",
        "activelyTrading": "actively_trading",
        "isEtf": "is_etf",
        "isFund": "is_fund",
    }
    for src, dst in rename_map.items():
        if src in out.columns and dst not in out.columns:
            out[dst] = out[src]
    if "symbol" in out.columns:
        out["symbol"] = out["symbol"].astype(str).str.strip().str.upper()
        out = out.loc[out["symbol"].ne("")].copy()
    if "market_cap" in out.columns:
        out["market_cap"] = pd.to_numeric(out["market_cap"], errors="coerce")
    return out


def fetch_openbb_fmp_screener(exchange: str) -> pd.DataFrame:
    from openbb import obb

    configure_openbb_credentials()
    result = obb.equity.screener(
        provider=PROVIDER,
        mktcap_min=SCREEN_MARKET_CAP_MIN,
        country=SCREEN_COUNTRY,
        exchange=exchange,
        is_etf=False,
        is_fund=False,
        is_active=True,
        all_share_classes=False,
        limit=SCREEN_LIMIT,
    )
    return _normalize_screener_frame(result.to_df())


def fetch_screened_universe() -> tuple[pd.DataFrame, pd.DataFrame]:
    frames: list[pd.DataFrame] = []
    fetch_rows: list[dict[str, object]] = []
    for exchange in SCREEN_EXCHANGES:
        try:
            frame = fetch_openbb_fmp_screener(exchange)
        except Exception as exc:
            fetch_rows.append({"exchange": exchange, "source": "openbb:fmp", "rows": 0, "status": type(exc).__name__})
            continue
        frame = frame.copy()
        frame["requested_exchange"] = exchange
        frames.append(frame)
        fetch_rows.append({"exchange": exchange, "source": "openbb:fmp", "rows": len(frame), "status": "ok"})

    raw = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if raw.empty:
        return raw, pd.DataFrame(fetch_rows)

    raw = raw.drop_duplicates("symbol").sort_values("market_cap", ascending=False).reset_index(drop=True)
    rows: list[dict[str, object]] = []
    for record in raw.where(raw.notna(), None).to_dict(orient="records"):
        symbol = str(record.get("symbol") or "").strip().upper()
        history_ok, history_reason = has_required_warehouse_history(symbol)
        row = dict(record)
        row["warehouse_history_ok"] = history_ok
        row["warehouse_history_reason"] = history_reason
        rows.append(row)

    screened = pd.DataFrame(rows)
    eligible = screened.loc[screened["warehouse_history_ok"]].copy()
    eligible = eligible.sort_values("market_cap", ascending=False).reset_index(drop=True)
    return eligible, pd.DataFrame(fetch_rows)


universe, screener_fetch_log = fetch_screened_universe()
ANALYSIS_SYMBOLS = tuple(universe["symbol"].astype(str)) if not universe.empty else tuple()

if not ANALYSIS_SYMBOLS:
    raise RuntimeError("OpenBB/FMP screener returned no warehouse-eligible symbols for the configured filters.")

display(screener_fetch_log)
display(universe[["symbol", "name", "market_cap", "exchange", "country", "is_etf", "is_fund", "actively_trading"]])
display(Markdown(f"> Selected {len(ANALYSIS_SYMBOLS):,} symbols from `{ANALYSIS_LABEL}`. Screening uses current OpenBB/FMP metadata; features use stored Quant Warehouse history only."))


,exchange,source,rows,status
0,NASDAQ,openbb:fmp,11,ok
1,NYSE,openbb:fmp,3,ok


,symbol,name,market_cap,exchange,country,is_etf,is_fund,actively_trading
0,NVDA,NVIDIA Corporation,4722368370000,NASDAQ,US,False,False,True
1,GOOGL,Alphabet Inc.,4277350879119,NASDAQ,US,False,False,True
2,GOOG,Alphabet Inc.,4269161251258,NASDAQ,US,False,False,True
3,AAPL,Apple Inc.,4138015679440,NASDAQ,US,False,False,True
4,MSFT,Microsoft Corporation,2737896445100,NASDAQ,US,False,False,True
5,AMZN,"Amazon.com, Inc.",2583209994000,NASDAQ,US,False,False,True
6,AVGO,Broadcom Inc.,1771960671000,NASDAQ,US,False,False,True
7,TSLA,"Tesla, Inc.",1546755724800,NASDAQ,US,False,False,True
8,META,"Meta Platforms, Inc.",1428116945204,NASDAQ,US,False,False,True
9,MU,"Micron Technology, Inc.",1293467779200,NASDAQ,US,False,False,True


> Selected 13 symbols from `OpenBB/FMP screened US >= $1T universe`. Screening uses current OpenBB/FMP metadata; features use stored Quant Warehouse history only.

## Feature Families

The feature families are `income_mcap`, `balance_mcap`, and `cash_mcap`. Each family contains the market-cap-denominated statement columns that exist in the stored FMP sections. The goal is to keep the family count low while still letting each source section contribute its full column width.

In [2]:

@dataclass(frozen=True)
class FeatureSpec:
    feature: str
    family: str
    source_section: str
    source_column: str


def _slice(frame: pd.DataFrame, start: str | None, end: str | None) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    out = frame.copy()
    out.index = pd.to_datetime(out.index, errors="coerce")
    out = out.loc[out.index.notna()].sort_index()
    if start is not None:
        out = out.loc[out.index >= pd.Timestamp(start)]
    if end is not None:
        out = out.loc[out.index <= pd.Timestamp(end)]
    return out


def _numeric_frame(frame: pd.DataFrame) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    out = frame.apply(pd.to_numeric, errors="coerce")
    return out.replace([np.inf, -np.inf], np.nan)


def _align_statement(frame: pd.DataFrame, daily_index: pd.DatetimeIndex, *, lag_days: int = FILING_LAG_DAYS) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame(index=daily_index)
    sparse = _numeric_frame(frame)
    sparse.index = pd.DatetimeIndex(pd.to_datetime(sparse.index, errors="coerce")).normalize() + pd.Timedelta(days=int(lag_days))
    sparse = sparse.loc[sparse.index.notna()].sort_index()
    sparse = sparse.loc[~sparse.index.duplicated(keep="last")]
    return sparse.reindex(daily_index, method="ffill")


def _safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator.divide(denominator.replace(0.0, np.nan))


def load_symbol_inputs(symbol: str) -> dict[str, pd.DataFrame]:
    return {
        "prices": _slice(wh.read_prices(symbol, provider=PROVIDER), START_DATE, END_DATE),
        "historical_market_cap": _slice(wh.read_fundamentals(symbol, section="historical_market_cap", provider=PROVIDER), START_DATE, END_DATE),
        "income": _slice(wh.read_fundamentals(symbol, section="income", provider=PROVIDER), None, END_DATE),
        "balance": _slice(wh.read_fundamentals(symbol, section="balance", provider=PROVIDER), None, END_DATE),
        "cash": _slice(wh.read_fundamentals(symbol, section="cash", provider=PROVIDER), None, END_DATE),
    }


def build_symbol_panel(symbol: str) -> tuple[pd.DataFrame, list[FeatureSpec], dict[str, object]]:
    inputs = load_symbol_inputs(symbol)
    prices = inputs["prices"]
    mcap = inputs["historical_market_cap"]
    if prices.empty or mcap.empty or "close" not in prices.columns or "market_cap" not in mcap.columns:
        return pd.DataFrame(), [], {"symbol": symbol, "status": "missing_prices_or_market_cap"}

    close = pd.to_numeric(prices["close"], errors="coerce")
    daily_index = pd.DatetimeIndex(prices.index)
    daily_mcap = pd.to_numeric(mcap["market_cap"], errors="coerce").reindex(daily_index, method="ffill")
    panel = pd.DataFrame({"date": daily_index, "symbol": symbol, "close": close, "daily_market_cap": daily_mcap}, index=daily_index)
    specs: list[FeatureSpec] = []
    feature_frames: dict[str, pd.Series] = {}

    align_start = perf_counter()
    aligned_sources = {
        section: _align_statement(inputs[section], daily_index)
        for section in SOURCE_SECTIONS
    }
    align_seconds = perf_counter() - align_start

    section_column_counts = {}
    for section, family in FAMILY_BY_SECTION.items():
        aligned = aligned_sources[section]
        if aligned.empty:
            continue
        for column in aligned.columns:
            if column.lower() in {"fiscal_year"}:
                continue
            series = pd.to_numeric(aligned[column], errors="coerce")
            if series.notna().sum() < MIN_OBS:
                continue
            feature = f"{family}__{column}"
            feature_frames[feature] = _safe_divide(daily_mcap, series.reindex(daily_index))
            specs.append(FeatureSpec(feature=feature, family=family, source_section=section, source_column=column))
        section_column_counts[section] = int(aligned.columns.size)

    if feature_frames:
        feature_panel = pd.DataFrame(feature_frames, index=daily_index)
        panel = pd.concat([panel, feature_panel], axis=1)

    for horizon in HORIZONS:
        panel[f"forward_return_{horizon}d"] = close.shift(-int(horizon)) / close - 1.0

    diagnostics = {
        "symbol": symbol,
        "status": "ok",
        "price_rows": len(prices),
        "market_cap_rows": len(mcap),
        "income_rows": len(inputs["income"]),
        "balance_rows": len(inputs["balance"]),
        "cash_rows": len(inputs["cash"]),
        "income_columns": section_column_counts.get("income", 0),
        "balance_columns": section_column_counts.get("balance", 0),
        "cash_columns": section_column_counts.get("cash", 0),
        "feature_count": len(specs),
        "source_align_seconds": align_seconds,
    }
    return panel.reset_index(drop=True), specs, diagnostics


In [3]:

panel_build_start = perf_counter()
frames: list[pd.DataFrame] = []
all_specs: list[FeatureSpec] = []
diagnostics: list[dict[str, object]] = []
for symbol in ANALYSIS_SYMBOLS:
    frame, specs, diag = build_symbol_panel(symbol)
    diagnostics.append(diag)
    if not frame.empty:
        frames.append(frame)
        all_specs.extend(specs)

panel = pd.concat(frames, ignore_index=True).sort_values(["date", "symbol"]).reset_index(drop=True)
feature_metadata = pd.DataFrame([spec.__dict__ for spec in all_specs]).drop_duplicates().sort_values(["family", "feature"]).reset_index(drop=True)
diagnostics_df = pd.DataFrame(diagnostics)
feature_cols = feature_metadata["feature"].tolist()
run_timings["panel_build_seconds"] = perf_counter() - panel_build_start
run_timings["source_align_seconds"] = float(diagnostics_df.get("source_align_seconds", pd.Series(dtype="float64")).sum())

print({
    "symbols": sorted(panel["symbol"].unique()),
    "rows": len(panel),
    "features": len(feature_cols),
    "start": str(panel["date"].min().date()),
    "end": str(panel["date"].max().date()),
    "panel_build_seconds": round(run_timings["panel_build_seconds"], 2),
    "source_align_seconds": round(run_timings["source_align_seconds"], 2),
})
display(diagnostics_df)
display(feature_metadata.groupby("family").size().rename("feature_count").reset_index())
display(feature_metadata.head(40))


{'symbols': ['AAPL', 'AMZN', 'AVGO', 'BRK-A', 'BRK-B', 'GOOG', 'GOOGL', 'LLY', 'META', 'MSFT', 'MU', 'NVDA', 'TSLA'], 'rows': 27690, 'features': 123, 'start': '2018-01-02', 'end': '2026-06-24', 'panel_build_seconds': 0.62, 'source_align_seconds': 0.06}


,symbol,status,price_rows,market_cap_rows,income_rows,balance_rows,cash_rows,income_columns,balance_columns,cash_columns,feature_count,source_align_seconds
0,NVDA,ok,2130,2127,109,109,109,32,54,40,123,0.0079
1,GOOGL,ok,2130,2127,97,90,93,32,54,40,123,0.0042
2,GOOG,ok,2130,2127,97,90,93,32,54,40,123,0.0041
3,AAPL,ok,2130,2127,162,150,145,32,54,40,123,0.0043
4,MSFT,ok,2130,2127,162,151,147,32,54,40,123,0.0043
5,AMZN,ok,2130,2127,121,118,117,32,54,40,123,0.0042
6,AVGO,ok,2130,2127,72,70,72,32,54,40,123,0.0041
7,TSLA,ok,2130,2127,73,72,73,32,54,40,123,0.0041
8,META,ok,2130,2127,61,58,61,32,54,40,123,0.0041
9,MU,ok,2130,2127,163,153,148,32,54,40,123,0.0043


,family,feature_count
0,balance_mcap,53
1,cash_mcap,39
2,income_mcap,31


,feature,family,source_section,source_column
0,balance_mcap__account_payables,balance_mcap,balance,account_payables
1,balance_mcap__accounts_receivables,balance_mcap,balance,accounts_receivables
2,balance_mcap__accrued_expenses,balance_mcap,balance,accrued_expenses
3,balance_mcap__accumulated_other_comprehensive_...,balance_mcap,balance,accumulated_other_comprehensive_income_loss
4,balance_mcap__additional_paid_in_capital,balance_mcap,balance,additional_paid_in_capital
5,balance_mcap__capital_lease_obligations,balance_mcap,balance,capital_lease_obligations
6,balance_mcap__capital_lease_obligations_current,balance_mcap,balance,capital_lease_obligations_current
7,balance_mcap__capital_lease_obligations_non_cu...,balance_mcap,balance,capital_lease_obligations_non_current
8,balance_mcap__cash_and_cash_equivalents,balance_mcap,balance,cash_and_cash_equivalents
9,balance_mcap__cash_and_short_term_investments,balance_mcap,balance,cash_and_short_term_investments


## Coverage And Sanity Checks

This section verifies that the market cap series is daily, the statement sources are aligned as-of filing lag, and the family widths are what we expect for a first pass.

In [4]:

coverage_rows = []
for symbol, group in panel.groupby("symbol"):
    group = group.sort_values("date")
    mcap = group.dropna(subset=["daily_market_cap"])
    coverage_rows.append({
        "symbol": symbol,
        "rows": len(group),
        "mcap_days": len(mcap),
        "mcap_start": str(mcap["date"].min().date()) if not mcap.empty else None,
        "mcap_end": str(mcap["date"].max().date()) if not mcap.empty else None,
        "features": int(group.filter(regex=r"^(income_mcap|balance_mcap|cash_mcap)__").shape[1]),
    })
coverage_df = pd.DataFrame(coverage_rows).sort_values("rows", ascending=False)
display(coverage_df)


,symbol,rows,mcap_days,mcap_start,mcap_end,features
0,AAPL,2130,2130,2018-01-02,2026-06-24,123
1,AMZN,2130,2130,2018-01-02,2026-06-24,123
2,AVGO,2130,2130,2018-01-02,2026-06-24,123
3,BRK-A,2130,2130,2018-01-02,2026-06-24,123
4,BRK-B,2130,2130,2018-01-02,2026-06-24,123
5,GOOG,2130,2130,2018-01-02,2026-06-24,123
6,GOOGL,2130,2130,2018-01-02,2026-06-24,123
7,LLY,2130,2130,2018-01-02,2026-06-24,123
8,META,2130,2130,2018-01-02,2026-06-24,123
9,MSFT,2130,2130,2018-01-02,2026-06-24,123


## Evaluation

The question here is whether statement-level market-cap ratios have any usable return relationship in this screened large-cap universe. I’m using forward returns at 20, 60, and 120 trading days, with daily cross-sectional rank IC and top-minus-bottom spread diagnostics.

In [5]:

def _rank_2d_nan(values: np.ndarray) -> np.ndarray:
    out = np.full(values.shape, np.nan, dtype="float32")
    for i in range(values.shape[0]):
        row = values[i]
        valid = np.isfinite(row)
        count = int(valid.sum())
        if count == 0:
            continue
        order = np.argsort(row[valid], kind="mergesort")
        ranks = np.empty(count, dtype="float32")
        ranks[order] = np.arange(1, count + 1, dtype="float32")
        out[i, np.flatnonzero(valid)] = ranks
    return out


def _rank_3d_by_symbol(values: np.ndarray) -> np.ndarray:
    days, symbols, features = values.shape
    flat = values.transpose(0, 2, 1).reshape(days * features, symbols)
    ranked = _rank_2d_nan(flat)
    return ranked.reshape(days, features, symbols).transpose(0, 2, 1)


def _mean_center_nan(values: np.ndarray, axis: int) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean = np.nanmean(values, axis=axis, keepdims=True)
    return values - mean


def _build_eval_arrays(features: list[str]) -> tuple[np.ndarray, dict[int, np.ndarray], pd.DatetimeIndex, list[str]]:
    dates = pd.DatetimeIndex(sorted(panel["date"].dropna().unique()))
    symbols = sorted(panel["symbol"].dropna().unique())
    feature_values = np.full((len(dates), len(symbols), len(features)), np.nan, dtype="float32")
    for feature_idx, feature in enumerate(features):
        pivot = panel.pivot(index="date", columns="symbol", values=feature).reindex(index=dates, columns=symbols)
        feature_values[:, :, feature_idx] = pivot.to_numpy(dtype="float32")

    feature_scores = feature_values
    returns_by_horizon: dict[int, np.ndarray] = {}
    for horizon in HORIZONS:
        pivot = panel.pivot(index="date", columns="symbol", values=f"forward_return_{horizon}d").reindex(index=dates, columns=symbols)
        returns_by_horizon[horizon] = pivot.to_numpy(dtype="float32")
    return feature_scores, returns_by_horizon, dates, symbols


def evaluate_array(features: list[str], horizons: tuple[int, ...]) -> pd.DataFrame:
    build_start = perf_counter()
    feature_scores, returns_by_horizon, dates, symbols = _build_eval_arrays(features)
    run_timings["eval_array_build_seconds"] = perf_counter() - build_start

    rank_start = perf_counter()
    feature_ranks = _rank_3d_by_symbol(feature_scores)
    run_timings["feature_rank_seconds"] = perf_counter() - rank_start

    rows = []
    for horizon in horizons:
        horizon_start = perf_counter()
        returns = returns_by_horizon[horizon]
        return_ranks = _rank_2d_nan(returns)

        centered_features = _mean_center_nan(feature_ranks, axis=1)
        centered_returns = _mean_center_nan(return_ranks, axis=1)
        valid = np.isfinite(centered_features) & np.isfinite(centered_returns[:, :, None])
        cf = np.where(valid, centered_features, 0.0)
        cr = np.where(np.isfinite(centered_returns), centered_returns, 0.0)
        numerator = np.einsum("dsf,ds->df", cf, cr, optimize=True)
        feature_ss = np.einsum("dsf,dsf->df", cf, cf, optimize=True)
        return_ss = np.einsum("ds,ds->d", cr, cr, optimize=True)
        denominator = np.sqrt(feature_ss * return_ss[:, None])
        with np.errstate(invalid="ignore", divide="ignore"):
            daily_ic = numerator / denominator
        daily_ic[denominator == 0] = np.nan

        valid_counts = np.isfinite(feature_scores).sum(axis=1).astype("float32")
        n_by_feature = np.where(valid_counts < 4, 1.0, 2.0)
        top_mask = feature_ranks > (valid_counts[:, None, :] - n_by_feature[:, None, :])
        bottom_mask = feature_ranks <= n_by_feature[:, None, :]
        returns_3d = returns[:, :, None]
        top_returns = np.where(top_mask & np.isfinite(returns_3d), returns_3d, np.nan)
        bottom_returns = np.where(bottom_mask & np.isfinite(returns_3d), returns_3d, np.nan)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            daily_spread = np.nanmean(top_returns, axis=1) - np.nanmean(bottom_returns, axis=1)

        for feature_idx, feature in enumerate(features):
            ic_series = daily_ic[:, feature_idx]
            spread_series = daily_spread[:, feature_idx]
            rows.append({
                "method": "array_evaluator",
                "feature": feature,
                "horizon": horizon,
                "mean_daily_rank_ic": float(np.nanmean(ic_series)),
                "median_daily_rank_ic": float(np.nanmedian(ic_series)),
                "rank_ic_hit_rate": float(np.nanmean(ic_series > 0)),
                "mean_spread_bps": float(np.nanmean(spread_series) * 10_000.0),
                "median_spread_bps": float(np.nanmedian(spread_series) * 10_000.0),
                "observations": int(np.isfinite(ic_series).sum()),
                "seconds_per_feature_horizon": float((perf_counter() - horizon_start) / max(len(features), 1)),
            })
    return pd.DataFrame(rows)


run_start = perf_counter()
results_df = evaluate_array(feature_cols, HORIZONS)
run_timings["evaluation_seconds"] = perf_counter() - run_start
results_df = results_df.merge(feature_metadata, on="feature", how="left")

display(pd.DataFrame([
    {"method": "array_evaluator", "seconds": run_timings["evaluation_seconds"], "features": len(feature_cols), "horizons": len(HORIZONS), "seconds_per_feature_horizon": run_timings["evaluation_seconds"] / max(len(feature_cols) * len(HORIZONS), 1)},
]))

display(results_df.sort_values(["horizon", "mean_daily_rank_ic"], ascending=[True, False]).head(30))
summary_by_family = results_df.groupby(["horizon", "family"]).agg(
    features=("feature", "nunique"),
    mean_rank_ic=("mean_daily_rank_ic", "mean"),
    median_rank_ic=("median_daily_rank_ic", "median"),
    median_spread_bps=("median_spread_bps", "median"),
    positive_ic_share=("rank_ic_hit_rate", "mean"),
).reset_index()
display(summary_by_family.sort_values(["horizon", "mean_rank_ic"], ascending=[True, False]))


/tmp/ipykernel_1758186/3545182016.py:92: RuntimeWarning: Mean of empty slice
  "mean_daily_rank_ic": float(np.nanmean(ic_series)),
/tmp/ipykernel_1758186/3545182016.py:93: RuntimeWarning: All-NaN slice encountered
  "median_daily_rank_ic": float(np.nanmedian(ic_series)),
/tmp/ipykernel_1758186/3545182016.py:95: RuntimeWarning: Mean of empty slice
  "mean_spread_bps": float(np.nanmean(spread_series) * 10_000.0),
/tmp/ipykernel_1758186/3545182016.py:96: RuntimeWarning: All-NaN slice encountered
  "median_spread_bps": float(np.nanmedian(spread_series) * 10_000.0),
/tmp/ipykernel_1758186/3545182016.py:92: RuntimeWarning: Mean of empty slice
  "mean_daily_rank_ic": float(np.nanmean(ic_series)),
/tmp/ipykernel_1758186/3545182016.py:93: RuntimeWarning: All-NaN slice encountered
  "median_daily_rank_ic": float(np.nanmedian(ic_series)),
/tmp/ipykernel_1758186/3545182016.py:95: RuntimeWarning: Mean of empty slice
  "mean_spread_bps": float(np.nanmean(spread_series) * 10_000.0),
/tmp/ipykernel_17

,method,seconds,features,horizons,seconds_per_feature_horizon
0,array_evaluator,1.2504,123,3,0.0034


,method,feature,horizon,mean_daily_rank_ic,median_daily_rank_ic,rank_ic_hit_rate,mean_spread_bps,median_spread_bps,observations,seconds_per_feature_horizon,family,source_section,source_column
34,array_evaluator,balance_mcap__property_plant_equipment_net,20,0.0726,0.0934,0.5953,171.2400,146.7086,2110,0.0005,balance_mcap,balance,property_plant_equipment_net
19,array_evaluator,balance_mcap__long_term_investments,20,0.0655,0.0816,0.5728,331.6307,237.4490,2110,0.0005,balance_mcap,balance,long_term_investments
94,array_evaluator,income_mcap__cost_of_revenue,20,0.0589,0.0714,0.5657,147.0656,138.2464,2110,0.0005,income_mcap,income,cost_of_revenue
109,array_evaluator,income_mcap__net_income_from_discontinued_oper...,20,0.0580,0.0524,0.0573,53.2730,0.0000,226,0.0005,income_mcap,income,net_income_from_discontinued_operations
9,array_evaluator,balance_mcap__cash_and_short_term_investments,20,0.0553,0.0714,0.5587,265.6920,193.6349,2110,0.0005,balance_mcap,balance,cash_and_short_term_investments
0,array_evaluator,balance_mcap__account_payables,20,0.0552,0.0604,0.5474,148.0980,133.9889,2110,0.0005,balance_mcap,balance,account_payables
44,array_evaluator,balance_mcap__total_equity,20,0.0541,0.0714,0.5714,75.0684,86.5494,2110,0.0005,balance_mcap,balance,total_equity
93,array_evaluator,income_mcap__cost_and_expenses,20,0.0538,0.0824,0.5681,142.7644,171.5942,2110,0.0005,income_mcap,income,cost_and_expenses
50,array_evaluator,balance_mcap__total_payables,20,0.0531,0.0440,0.5376,79.1152,61.2364,2110,0.0005,balance_mcap,balance,total_payables
51,array_evaluator,balance_mcap__total_stockholders_equity,20,0.0529,0.0714,0.5695,86.2902,77.7055,2110,0.0005,balance_mcap,balance,total_stockholders_equity


,horizon,family,features,mean_rank_ic,median_rank_ic,median_spread_bps,positive_ic_share
0,20,balance_mcap,53,0.0211,0.0403,78.5238,0.4849
2,20,income_mcap,31,0.0115,0.0217,42.9527,0.4633
1,20,cash_mcap,39,-0.0017,0.0000,1.6578,0.4654
3,60,balance_mcap,53,0.0427,0.0670,283.3012,0.4971
5,60,income_mcap,31,0.0155,0.0275,66.8080,0.4524
4,60,cash_mcap,39,0.0038,0.0110,42.7906,0.4573
6,120,balance_mcap,53,0.0424,0.0628,409.1479,0.4854
8,120,income_mcap,31,0.0259,0.0165,173.7834,0.4435
7,120,cash_mcap,39,0.0123,0.0165,76.7298,0.4502


## Analysis

In [6]:

def _fmt_seconds(value: float) -> str:
    if pd.isna(value):
        return "nan"
    return f"{value:,.2f}s"

best_by_horizon = summary_by_family.sort_values(["horizon", "mean_rank_ic"], ascending=[True, False]).groupby("horizon").head(1)
family_counts = feature_metadata.groupby("family").size().rename("feature_count").reset_index()
current_total = run_timings.get("panel_build_seconds", 0.0) + run_timings.get("evaluation_seconds", 0.0)

analysis_md = "\n".join([
    f"### {ANALYSIS_LABEL} Statement-Level Market Cap Ratio Analysis",
    "",
    f"The notebook completed on {panel['symbol'].nunique()} symbols, {len(panel):,} symbol-days, and {len(feature_cols)} market-cap-denominated statement features across the {len(family_counts)} families `income_mcap`, `balance_mcap`, and `cash_mcap`.",
    f"Panel construction took {_fmt_seconds(run_timings.get('panel_build_seconds', np.nan))}; evaluation took {_fmt_seconds(run_timings.get('evaluation_seconds', np.nan))}; total core runtime was {_fmt_seconds(current_total)}.",
    "",
    "#### Family Width",
    f"- `income_mcap`: {int(family_counts.loc[family_counts['family'].eq('income_mcap'), 'feature_count'].iloc[0])} features",
    f"- `balance_mcap`: {int(family_counts.loc[family_counts['family'].eq('balance_mcap'), 'feature_count'].iloc[0])} features",
    f"- `cash_mcap`: {int(family_counts.loc[family_counts['family'].eq('cash_mcap'), 'feature_count'].iloc[0])} features",
    "",
    "#### Coverage",
    f"- Screened universe: {len(ANALYSIS_SYMBOLS)} symbols from OpenBB/FMP with `mktcap_min={SCREEN_MARKET_CAP_MIN:,}` and `country='{SCREEN_COUNTRY}'`.",
    "- Only symbols with stored historical prices, market cap, income, balance, and cash data were retained.",
    "- The family widths come from the statement sections themselves; daily market-cap adjustment changes the values, not the number of source columns.",
    "",
    "#### Result",
    *[
        f"- {row.horizon}d best family: `{row.family}` with mean daily rank IC {row.mean_rank_ic:.4f} and median spread {row.median_spread_bps:,.1f} bps"
        for row in best_by_horizon.itertuples(index=False)
    ],
    "",
    "The short version is that the pipeline works, the naming is compact, and the statement-level market-cap ratios are easy to group into three stable families. The signal result is what matters next: if one family clearly dominates, keep it; if not, the structure is still useful because it preserves the raw statement widths while keeping the daily layer derived and separate from source storage.",
])

display(Markdown(analysis_md))


### OpenBB/FMP screened US >= $1T universe Statement-Level Market Cap Ratio Analysis

The notebook completed on 13 symbols, 27,690 symbol-days, and 123 market-cap-denominated statement features across the 3 families `income_mcap`, `balance_mcap`, and `cash_mcap`.
Panel construction took 0.62s; evaluation took 1.25s; total core runtime was 1.87s.

#### Family Width
- `income_mcap`: 31 features
- `balance_mcap`: 53 features
- `cash_mcap`: 39 features

#### Coverage
- Screened universe: 13 symbols from OpenBB/FMP with `mktcap_min=1,000,000,000,000` and `country='US'`.
- Only symbols with stored historical prices, market cap, income, balance, and cash data were retained.
- The family widths come from the statement sections themselves; daily market-cap adjustment changes the values, not the number of source columns.

#### Result
- 20d best family: `balance_mcap` with mean daily rank IC 0.0211 and median spread 78.5 bps
- 60d best family: `balance_mcap` with mean daily rank IC 0.0427 and median spread 283.3 bps
- 120d best family: `balance_mcap` with mean daily rank IC 0.0424 and median spread 409.1 bps

The short version is that the pipeline works, the naming is compact, and the statement-level market-cap ratios are easy to group into three stable families. The signal result is what matters next: if one family clearly dominates, keep it; if not, the structure is still useful because it preserves the raw statement widths while keeping the daily layer derived and separate from source storage.